# Reproducing Hart et al. (2024) from dandiset 000773

Paper. Hart T, Lopes LE, Frank DD, Kronauer DJC (2024). Pheromone representation in the ant
antennal lobe changes with age. *Current Biology* 34, 3233-3240.
[doi:10.1016/j.cub.2024.05.031](https://doi.org/10.1016/j.cub.2024.05.031). The claims file was
written from the accepted author manuscript in PubMed Central (PMC11265976, NIHMS1997610), main
text, legends and STAR Methods, with the supplement (Figures S1 to S4, Table S1).

Dandiset. [DANDI:000773](https://dandiarchive.org/dandiset/000773/0.261004.1511), version
0.261004.1511, the only published version. The paper cites the dandiset without a version, so the
most recent published version is pinned; the draft has the same 15,246 assets and size, and was
last modified when this version was published (2026-10-04).

What the dandiset holds. The two-photon calcium imaging of Figures 2C to 4 and S2 to S4: 21
GCaMP6s ants (14 at 48% and 7 at 3% odor concentration), each with 22 trials (seven odors three
times, and one paraffin oil trial) of 33 planes; one NWB file per plane and trial. It does not hold
the behaviour of Figure 1 and S1, the confocal stacks of Figure 2A and 2B, or the 90-day-old ants
of Figure 3C and 3D (DANDI 000467, read here for reading (a) of
fig3d-pg-60-vs-90-days-same).

Outside data. The paper's supplement Data S1 (behavioural counts, MMC2.csv) and Data S2
(per-glomerulus fold-change traces, MMC3.xlsx), from the PubMed Central Open Access copy
PMC11265976.1, pinned by MD5. Data S1 is the data of the five behaviour claims (`data: elsewhere`).
Data S2 is used for three things: to name the ants and trials of the dandiset in the paper's
terms, to find where each focal glomerulus is in each ant (the paper drew these ROIs by hand and
gives no coordinates; see the link claim `data-s2-glomerulus-link`), and as the 90-day values of
reading (b) of `fig3d-pg-60-vs-90-days-same`. Every AUC, map and count tested here is computed
from the dandiset's pixels.

| Claim | Data | Tested here |
|---|---|---|
| fig1c-old-faster-alarm, fig1c-vehicle-single-curve, fig1b-alarm-stimuli-similar, figs1-heptanone-separate-curves, figs1-heptanol-blend-single-curve | Data S1 (elsewhere) | yes |
| fig2b-gcamp-baseline-unchanged | Brain Image Library (elsewhere) | no, blocked: the download server could not be reached (three attempts) |
| fig2d-whole-al-alarm-no-age-effect, figs3b-whole-al-general-decline, figs3b-whole-al-other-no-age-effect | dandiset | yes |
| fig2e-heptanone-breadth-falls, fig2-alarm-sparse, figs3c-general-breadth-no-age-effect | dandiset | run with the claims file's automated ROIs; all three blocked as methods underspecified |
| data-s2-glomerulus-link (link claim added here) | dandiset and Data S2 | yes |
| figs2a, fig3b, fig3d-pg-stronger-old, figs4a, fig4 (two claims), figs4d, pg-opposite-modulation | dandiset, ROI positions from Data S2 | yes |
| fig3d-pg-60-vs-90-days-same | dandiset, DANDI 000467 and Data S2 (with inputs) | both readings run; blocked as methods underspecified |

The analysis streams about 460 GB (frames 1-5 and 9-14 of every plane file, the middle plane of
each 48% trial, then single rows of the planes around each focal glomerulus, and the same for the
eight 90-day ants of DANDI 000467) and takes about 40 minutes (36 to 44 in the runs so far).

Revision after the first verification. The focal glomeruli are now also placed without the trials
that are then tested (leave one odor out), which is the main measure, and every placement is
compared with its chance level; reading (b) of the glomerulus fold change leaves out pixels at
background level; the whole-AL measure's offset is shown; and the 90-day placements are tested
against chance.

Revision after the second verification. The landing page of the Brain Image Library dataset is
read and its notices printed; reading (a) of fig3d-pg-60-vs-90-days-same also compares the 60-day
values at the held-out placements with the 90-day values; and the text is corrected where the
round 2 verifier found it unsupported by the outputs.

In [ ]:
import hashlib
import io
import time
import warnings
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import antal

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
pd.set_option("display.float_format", "{:.4g}".format)
warnings.filterwarnings("ignore", category=RuntimeWarning)
started = time.time()

ANTS = None   # all ants

assets = antal.list_assets()
trials = antal.trial_table(assets)
if ANTS is not None:
    trials = trials[trials.subject.isin(ANTS)].reset_index(drop=True)
print(f"dandiset {antal.DANDISET_ID} version {antal.DANDISET_VERSION}: {len(assets)} assets, "
      f"{assets['size'].sum() / 1e12:.2f} TB; {len(trials)} trials of {trials.subject.nunique()} ants")

# one file per ant for its metadata
first = trials.groupby("subject").blobs.first()
with ThreadPoolExecutor(8) as ex:
    meta = dict(zip(first.index, ex.map(lambda b: antal.nwb_metadata(b[16]), first)))
meta = pd.DataFrame(meta).T
meta["age_days"] = meta.age.str.extract(r"P(\d+)D")[0].astype(int)
meta["cohort"] = np.where(meta.age_days < 40, "young", "old")
conc = trials[trials.odor != "paraffin-oil"].groupby("subject").conc.first()
meta["conc"] = conc
trials["cohort"] = trials.subject.map(meta.cohort)

## Data check

In [ ]:
print("Ants by concentration and cohort (ages from the NWB subject metadata, days post eclosion):")
print(meta.groupby(["conc", "cohort"]).age_days.agg(["count", "min", "max"]).to_string())
print("\nPer ant: age, trials per odor, planes per trial")
tab = trials.pivot_table(index="subject", columns="odor", values="t", aggfunc="count").fillna(0).astype(int)
tab.insert(0, "age_days", meta.age_days)
tab.insert(1, "conc", meta.conc)
tab["planes"] = trials.groupby("subject").n_planes.unique().map(lambda v: ",".join(map(str, v)))
print(tab.to_string())
print("\nOdor names in the file names:", assets.odor.value_counts().to_dict())
print("Plane depths (um):", sorted(assets.z.unique()))
print("Every trial has the 33 planes 0 to 160 um:",
      bool(trials.z.map(lambda z: z == list(range(0, 165, 5))).all()))
print("Asset sizes (bytes):", assets["size"].unique())

Ages agree with the Methods (young 15 to 20, old 60 to 61 days post eclosion) and the counts with
the Figure S3B legend (48%: 7 young and 7 old; 3%: 3 young and 4 old). One trial of one ant spells
ethylpyrazine "ethylyprazine" in its 33 file names; it is read as ethylpyrazine (its position in the
odor sequence is that of ethylpyrazine; see the next table).

In [ ]:
odd = assets[assets.odor == "ethylyprazine"].drop_duplicates(["subject", "t"])
for _, r in odd.iterrows():
    seq = trials[trials.subject == r.subject].sort_values("t")[["t", "odor"]]
    print(r.subject, "trial", r.t, "is spelled 'ethylyprazine'; that ant's sequence:")
    print(seq.T.to_string(header=False))
print()
print("Metadata of one file per ant (first trial, plane 80 um):")
print(meta[["age", "species", "unit", "shape", "dtype", "rate_hz", "imaging_rate", "grid_spacing_um",
            "offset", "chunks", "compression"]].to_string())
print("\nSeries description:", meta.series_description.unique())
off = antal.check_offsets(trials.sample(6, random_state=0).blobs.map(lambda b: b[3]).tolist())
print("\nData offset and range reads checked against h5py on 6 files:")
print(off.to_string(index=False))

Each file holds 40 frames (volumes) of 512 x 512 raw fluorescence as float64, uncompressed and
contiguous, at the same byte offset, so frames are read by range; the range reads equal h5py's.
The pixel size, 0.289 um, matches 148 um over 512 pixels (Methods). The timing metadata disagree
with each other: the series description gives 40 cycles in 48.0 s (1.2 s per volume, as 33 planes
at 27.5 frames per second take), the series rate is 1.21 Hz (0.83 s per volume, the Methods' value),
and `imaging_rate` is 0.83, a period stored as a rate. All windows here are in frames, so this
changes no measure; Data S2 puts frames 1.2 s apart. The odor onset is not stored in the files,
only in the description ("odor stimulus triggered at 3s with duration 5s").

In [ ]:
s1_raw = antal.fetch(antal.DATA_S1, antal.DATA_S1_MD5)
s2_raw = antal.fetch(antal.DATA_S2, antal.DATA_S2_MD5)
print("Data S1", antal.DATA_S1, "MD5", hashlib.md5(s1_raw).hexdigest())
print("Data S2", antal.DATA_S2, "MD5", hashlib.md5(s2_raw).hexdigest())
s1 = pd.read_csv(io.BytesIO(s1_raw))
s2 = antal.load_data_s2(s2_raw)
print("\nData S1: trials per stimulus:", s1.groupby("stimulus").file_name.nunique().to_dict(),
      "; time points:", sorted(s1.time.unique()))
s2a = s2.groupby(["subject", "age", "conc"]).agg(glomeruli=("glomerulus", "nunique"),
                                                  trials=("trial", "nunique")).reset_index()
s2a["in_dandiset"] = s2a.subject.isin(meta.index)
s2a["dandiset_age_days"] = s2a.subject.map(meta.age_days)
s2a["dandiset_conc"] = s2a.subject.map(meta.conc)
print("\nData S2 ants (names normalised to the dandiset's subject names):")
print(s2a.to_string(index=False))

Data S2 names its ants by number and imaging date (for example a4-1-18-2023, also written
a4-1-18-23); the dandiset's subjects carry the same number and date (ant4-m1-d18-y2023). The 14
ants of Data S2 at 48% from 2023 are the 14 dandiset ants at 48%, and Data S2's age labels (2 weeks,
2 months) agree with the dandiset's ages. The 3% ants are not in Data S2. The 2022 ants of Data S2
are the 90-day-old ants of the earlier study.

The confocal stacks of Figure 2A and 2B (fig2b-gcamp-baseline-unchanged) are in the Brain Image
Library (doi:10.35077/g.1153). The next cell reads the dataset's landing page (the brains it lists
under download.brainimagelibrary.org, and its notices), then tries to reach that download server
three times, 30 s apart, with a 60 s timeout each.

In [ ]:
import html
import re
import requests
try:
    page = requests.get("https://doi.org/10.35077/g.1153", timeout=60)
    text = html.unescape(re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", page.text)))
    brains = sorted(set(re.findall(r"download\.brainimagelibrary\.org/\S+/(Two_\w+_Old/\w+_Brain_\d+)", text)))
    print("Landing page", page.url, page.status_code)
    print(f"Brains listed: {len(brains)} ({sum('Young' in b for b in brains)} young, {sum('Old_Brain' in b for b in brains)} old)")
    for notice in ("This repository is under review[^.]*\\.", "BIL data will be unavailable[^.]*\\."):
        print("Notice:", *re.findall(notice, text)[:1])
except Exception as e:
    print("Landing page not reachable,", type(e).__name__)
bil = "https://download.brainimagelibrary.org/3d/38/3d38636a9ef063d3/Two_Week_Old/Young_Brain_1/"
for attempt in range(1, 4):
    try:
        r = requests.get(bil, timeout=60)
        print(f"Brain Image Library download server, attempt {attempt}:", r.status_code, len(r.content), "bytes")
        break
    except Exception as e:
        print(f"Brain Image Library download server, attempt {attempt} (60 s timeout): not reachable,", type(e).__name__)
        time.sleep(30)


## The behaviour claims (Data S1)

The five behaviour claims use Data S1, which the dandiset does not hold. Per trial and time point
from -10 to 60 s, the proportion of young ants outside the nest over the colony's young (and the
same for old), as the paper's `alarm_behavior_calculations.py` computes it (claims file); Data S1
carries these proportions. The logistic of Table S1 (and the quadratic for the vehicle) is fit by
least squares to all per-trial points; the extra sum-of-squares F test compares one shared curve
with one curve per group. Starting values (YM 0.9, Y0 0.5, K 0.05) are a standard choice.

In [ ]:
b = s1[(s1.time >= -10) & (s1.time <= 60)]
P0 = [0.9, 0.5, 0.05]
alarm = b[b.stimulus != "pentane"]
def xy(df, col):
    return df.time.to_numpy(float), df[col].to_numpy(float)
def show(name, r, names=("YM", "Y0", "K")):
    print(f"{name}: F({r['df'][0]}, {r['df'][1]}) = {r['F']:.3f}, p = {r['p']:.2g}")
    for g, p in r["separate"].items():
        print(f"   {g:>14}: " + ", ".join(f"{n} {v:.4g}" for n, v in zip(names, p)))
    print(f"   {'shared':>14}: " + ", ".join(f"{n} {v:.4g}" for n, v in zip(names, r["shared"])))

In [ ]:
r_age = antal.extra_ss_test(antal.logistic, {"young": xy(alarm, "prop_outside_2wo"),
                                             "old": xy(alarm, "prop_outside_2mo")}, P0)
show("Pooled alarm stimuli, young against old", r_age)
tg = np.linspace(0, 60, 61)
yo, ol = (antal.logistic(tg, *r_age["separate"][k]) for k in ("young", "old"))
print(f"Old curve above young at every second from 0 to 60 s: {bool((ol > yo).all())} "
      f"(smallest old minus young {np.min(ol - yo):.3f})")
exp = {"YM young": (0.858, 0.081), "YM old": (0.914, 0.033), "Y0 young": (0.571, 0.046),
       "Y0 old": (0.663, 0.040), "K young": (0.070, 0.036), "K old": (0.099, 0.029)}
got = {"YM young": r_age["separate"]["young"][0], "YM old": r_age["separate"]["old"][0],
       "Y0 young": r_age["separate"]["young"][1], "Y0 old": r_age["separate"]["old"][1],
       "K young": r_age["separate"]["young"][2], "K old": r_age["separate"]["old"][2]}
print(pd.DataFrame({"expected": {k: v[0] for k, v in exp.items()}, "tolerance": {k: v[1] for k, v in exp.items()},
                    "measured": got}).assign(within=lambda d: (d.measured - d.expected).abs() <= d.tolerance).round(4).to_string())
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for a, df, title in ((ax[0], alarm, "pooled alarm stimuli"), (ax[1], b[b.stimulus == "pentane"], "vehicle")):
    for col, lab in (("prop_outside_2wo", "young"), ("prop_outside_2mo", "old")):
        m = df.groupby("time")[col].agg(["mean", "sem"])
        a.errorbar(m.index, m["mean"], m["sem"], marker="o", label=lab, capsize=2)
    a.set_title(title); a.set_xlabel("time (s)"); a.set_ylabel("proportion outside the nest"); a.set_ylim(0, 1)
ax[0].plot(tg, yo, "C0-"); ax[0].plot(tg, ol, "C1-"); ax[0].legend()
plt.tight_layout(); plt.show()
print("All plotted proportions lie in [0, 1]:", bool(b[["prop_outside_2wo", "prop_outside_2mo"]].stack().between(0, 1).all()))

**Result: reproduced.** Separate curves for young and old ants fit better than one shared curve,
F(3, 314) = 8.006, p = 3.7e-05 (Table S1: F(3, 314) = 8.006, p < 0.0001). All six parameters
are within tolerance (YM 0.858 and 0.914, Y0 0.571 and 0.663, K 0.070 and 0.099 for young and old).
Old ants are faster under both readings: K is higher for old ants (0.0986 against 0.0695), and the
old curve lies above the young one at every second from 0 to 60 s (by at least 0.061).

In [ ]:
veh = b[b.stimulus == "pentane"]
r_veh = antal.extra_ss_test(antal.quadratic, {"young": xy(veh, "prop_outside_2wo"),
                                              "old": xy(veh, "prop_outside_2mo")}, P0)
show("Vehicle (pentane), young against old, quadratic", r_veh, names=("B0", "B1", "B2"))

**Result: reproduced.** For the vehicle, the quadratic fit gives F(3, 90) = 2.617, p = 0.056, so
the single curve is not rejected at 0.05, as in Table S1 (F(3, 90) = 2.617, p = 0.056).

In [ ]:
r_stim = antal.extra_ss_test(antal.logistic, {s: xy(alarm[alarm.stimulus == s], "prop_outside_total")
                                              for s in ("4m3heptanone", "4m3heptanol", "blend")}, P0)
show("Three alarm stimuli, all ants", r_stim)

**Result: reproduced.** One curve for the three alarm stimuli is not rejected: F(6, 151) = 2.144,
p = 0.052 (Table S1: F(6, 151) = 2.144, p = 0.052).

In [ ]:
r_one = {}
for s in ("4m3heptanone", "4m3heptanol", "blend"):
    d = alarm[alarm.stimulus == s]
    r_one[s] = antal.extra_ss_test(antal.logistic, {"young": xy(d, "prop_outside_2wo"),
                                                    "old": xy(d, "prop_outside_2mo")}, P0)
    show(f"{s}, young against old", r_one[s])
    yo, ol = (antal.logistic(tg, *r_one[s]["separate"][k]) for k in ("young", "old"))
    print(f"   old above young at every second 0-60 s: {bool((ol > yo).all())}; "
          f"K old above K young: {r_one[s]['separate']['old'][2] > r_one[s]['separate']['young'][2]}")

**Result: figs1-heptanone-separate-curves reproduced; figs1-heptanol-blend-single-curve
reproduced.** For 4-methyl-3-heptanone separate curves are preferred, F(3, 90) = 6.377, p = 0.00057,
and the old curve lies above the young one at every second from 0 to 60 s (reading b; K is also
slightly higher for old ants, 0.0973 against 0.0949). For 4-methyl-3-heptanol (F(3, 106) = 1.979,
p = 0.12) and the blend (F(3, 106) = 2.529, p = 0.061) single curves are not rejected; in both the
old curve lies above the young one, the similar trend the legend describes. All F values equal
Table S1's to the third decimal.

## Streaming the imaging data

For every trial (including the paraffin oil trials, used as a control), frames 1-5 and 9-14 of all
33 planes are read; each frame is aligned to the plane's frames 1-5 by phase correlation; the peak
fold change map is made as in the Methods (minimum filter of radius 2, maximum projection through
the planes, clipped to [-1, 4]). For the 48% ants, the per-plane peak maps are also kept in 4 x 4
pixel blocks, to find the focal glomeruli.

In [ ]:
t0 = time.time()
keep = [(s, t) for s, t, o, c in zip(trials.subject, trials.t, trials.odor, trials.conc) if c == 0.48 and o != "paraffin-oil"]
maps = antal.all_volume_maps({(s, t): b for s, t, b in zip(trials.subject, trials.t, trials.blobs)},
                             keep_keys=keep, processes=4, progress=100)
print(f"{len(maps)} trials in {(time.time() - t0) / 60:.1f} min")
trials["key"] = list(zip(trials.subject, trials.t))

In [ ]:
sh = pd.DataFrame([{"subject": k[0], "t": k[1], "median_shift": np.median([np.median(p) for p in v["shifts"]]),
                    "planes_over_2px": int(sum(np.median(p) > 2 for p in v["shifts"])),
                    "frames_not_aligned": int(sum(np.sum(np.array(p) > antal.MAX_SHIFT) for p in v["shifts"]))}
                   for k, v in maps.items()])
print("Motion: per trial, the median over planes of the median frame shift (pixels):")
print(sh.median_shift.describe().round(2).to_string())
print("Trials with a median shift above 2 px:", int((sh.median_shift > 2).sum()), "of", len(sh))
print("Frames whose shift estimate exceeded", antal.MAX_SHIFT, "px and were left in place:",
      int(sh.frames_not_aligned.sum()), "of", len(sh) * 33 * 11)
print("\nPer ant, median shift over trials:")
print(sh.groupby("subject").median_shift.median().round(1).to_string())
h = pd.Series({k: hashlib.md5(v["raw_max"].tobytes()).hexdigest() for k, v in maps.items()})
print("\nTrials whose raw maximum projection is identical to another trial's:", int(h.duplicated(keep=False).sum()))

## The whole-AL outline

The Methods define "a single ROI around the boundary of the AL for each ant, based on a max
Z-projection of raw fluorescence", drawn by hand. Here it is taken from the mean over the ant's
trials of the raw maximum projection (frames 1-5) by Otsu's threshold, a standard choice (claims
file); the claims are also run with the outline shrunk and grown by 5 pixels.

In [ ]:
raw_ant = {s: np.mean([maps[k]["raw_max"] for k in trials[trials.subject == s].key], 0) for s in meta.index if (trials.subject == s).any()}
masks = {s: {g: antal.al_mask(r, grow=g) for g in (-5, 0, 5)} for s, r in raw_ant.items()}
# the background level of each ant (median of the raw maximum projection outside the AL outline),
# the F0 floor of reading (b) of the glomerulus fold change (antal.roi_trace)
FLOOR = {s: float(np.median(r[~masks[s][0]])) for s, r in raw_ant.items()}
area = pd.DataFrame({s: {f"grow {g}": m[g].sum() for g in m} for s, m in masks.items()}).T
area["area_um2 (grow 0)"] = (area["grow 0"] * 0.2889 ** 2).round(0)
print("AL outline area per ant (pixels):")
print(area.to_string())
print("\nBackground level outside the AL (raw units), per ant:", {s: round(v, 1) for s, v in FLOOR.items()})
print("Raw maximum projection inside the AL, median over ants of the per-ant 1st percentile:",
      round(float(np.median([np.percentile(raw_ant[s][masks[s][0]], 1) for s in raw_ant])), 1))
ants = list(raw_ant)
fig, axs = plt.subplots(3, 7, figsize=(21, 9))
for a, s in zip(axs.flat, ants):
    a.imshow(np.log1p(raw_ant[s]), cmap="gray"); a.contour(masks[s][0], [0.5], colors="r", linewidths=0.8)
    a.set_title(f"{s} ({meta.cohort[s]}, {meta.conc[s]:.0%})", fontsize=8); a.axis("off")
plt.tight_layout(); plt.show()

## Stimulus timing

The files carry no stimulus times. As a check that the odor arrives in the frames the Methods
assume (after frames 1-5, peak in frames 9-14), the whole-AL mean of the peak map is compared
between odor trials and the paraffin oil trial of each ant, and (below, after the glomerulus
traces) the frame-by-frame time course is shown.

In [ ]:
def al_mean(key, grow=0, field="peak"):
    return float(maps[key][field][masks[key[0]][grow]].mean())
trials["al_mean"] = [al_mean(k) for k in trials.key]
trials["al_mean_g-5"] = [al_mean(k, -5) for k in trials.key]
trials["al_mean_g+5"] = [al_mean(k, 5) for k in trials.key]
trials["al_mean_no_z0"] = [al_mean(k, 0, "peak_no_z0") for k in trials.key]
trials["al_mean_no_min"] = [al_mean(k, 0, "peak_no_min") for k in trials.key]
# the offset of each ant: the whole-AL mean of its paraffin oil trial, subtracted as a variant
par_offset = trials[trials.odor == "paraffin-oil"].set_index("subject").al_mean
trials["al_mean_minus_paraffin"] = trials.al_mean - trials.subject.map(par_offset)
print("Whole-AL mean peak fold change, by odor (all ants, median over trials):")
print(trials.groupby(["conc", "odor"]).al_mean.describe()[["count", "50%", "min", "max"]].round(3).to_string())
print("\nThe offset: paraffin oil trials, whole-AL mean with the minimum filter (the Methods' map) and without it:")
po = trials[trials.odor == "paraffin-oil"]
print(po[["al_mean", "al_mean_no_min"]].describe().loc[["count", "min", "50%", "max"]].round(3).to_string())
print("Per ant paraffin oil offset:", par_offset.round(3).to_dict())

## fig2d-whole-al-alarm-no-age-effect

> The mean calcium response across the entire AL to each alarm pheromone does not differ between
> young and old ants, at 3% or at 48%.

Per trial, the mean of the peak map inside the AL outline; per ant, the mean of its three trials of
the odor; Welch's t test of young against old for each pheromone and concentration.

In [ ]:
odor_trials = trials[trials.odor != "paraffin-oil"]
def per_ant(col="al_mean", df=None):
    df = odor_trials if df is None else df
    return df.groupby(["conc", "odor", "subject", "cohort"])[col].mean().reset_index()
def welch_table(pa, col, odors, concs=(0.03, 0.48)):
    rows = []
    for c in concs:
        for o in odors:
            d = pa[(pa.conc == c) & (pa.odor == o)]
            y, ol = d[d.cohort == "young"][col], d[d.cohort == "old"][col]
            if len(y) < 2 or len(ol) < 2:
                continue
            t, p = stats.ttest_ind(y, ol, equal_var=False)
            rows.append({"conc": c, "odor": o, "n_young": len(y), "n_old": len(ol), "mean_young": y.mean(),
                         "mean_old": ol.mean(), "t": t, "p": p})
    return pd.DataFrame(rows)
res2d = {}
for col in ("al_mean", "al_mean_g-5", "al_mean_g+5", "al_mean_no_z0"):
    res2d[col] = welch_table(per_ant(col), col, antal.ALARM)
    print(f"--- {col} ---")
    print(res2d[col].to_string(index=False))
pa = per_ant()
print("\nPer ant values (al_mean):")
print(pa[pa.odor.isin(antal.ALARM)].pivot_table(index=["conc", "cohort", "subject"], columns="odor", values="al_mean").round(3).to_string())

**Result: reproduced.** None of the four Welch tests finds an age difference in the whole-AL mean
peak fold change: at 3%, p = 0.78 (4-methyl-3-heptanone) and 0.065 (4-methyl-3-heptanol); at 48%,
p = 0.39 and 0.14. The outline shrunk or grown by 5 pixels, and the projection without the 0 um
plane, give the same (smallest p 0.061). The cohort means are all negative (-0.15 to -0.26): the
minimum filter makes the map of a non-responding AL negative, as the paraffin oil trials show
(median -0.23).

## figs3b-whole-al-general-decline and figs3b-whole-al-other-no-age-effect

> At 48%, 3-hexanone and isopropanol evoke broad AL responses in young but not old ants; mean peak
> responses across the AL fall more than 9-fold (3-hexanone) and more than 57-fold (isopropanol).

> The age decline of whole-AL responses is specific to 3-hexanone and isopropanol at 48%.

The measure has a negative offset (paraffin oil trials near -0.2, above). Two variants show what it
does to the ratios of cohort means: each ant's paraffin oil value subtracted from its odor values
(`al_mean_minus_paraffin`), and the map made without the minimum filter (`al_mean_no_min`).

In [ ]:
res3b = {}
for col in ("al_mean", "al_mean_g-5", "al_mean_g+5", "al_mean_no_z0", "al_mean_minus_paraffin", "al_mean_no_min"):
    r = welch_table(per_ant(col), col, antal.GENERAL)
    r["ratio_young_over_old"] = r.mean_young / r.mean_old
    res3b[col] = r
    print(f"--- {col} ---")
    print(r.to_string(index=False))
print("\nPer ant values at 48% (al_mean):")
print(pa[(pa.conc == 0.48) & pa.odor.isin(antal.GENERAL)].pivot_table(index=["cohort", "subject"], columns="odor", values="al_mean").round(3).to_string())
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
for a, c in zip(ax, (0.03, 0.48)):
    d = pa[pa.conc == c]
    for i, o in enumerate(antal.ODORS):
        for j, co in enumerate(("young", "old")):
            v = d[(d.odor == o) & (d.cohort == co)].al_mean
            a.scatter(np.full(len(v), i + (j - 0.5) * 0.3), v, color=f"C{j}", s=12, label=co if i == 0 else None)
    a.set_xticks(range(7)); a.set_xticklabels([o[:12] for o in antal.ODORS], rotation=30, fontsize=7)
    a.set_title(f"{c:.0%}"); a.set_ylabel("whole-AL mean peak fold change"); a.legend(fontsize=7)
plt.tight_layout(); plt.show()
print("Range of plotted values:", round(pa.al_mean.min(), 3), "to", round(pa.al_mean.max(), 3))

**Result: figs3b-whole-al-general-decline partially reproduced; figs3b-whole-al-other-no-age-effect
reproduced.** At 48% the whole-AL response falls with age for 3-hexanone (young 1.613, old 0.039,
t = 7.39, p = 2.0e-05) and isopropanol (young 0.860, old -0.151, p = 0.0085). The ratio of cohort
means is 41.0 for 3-hexanone, above the 8.1 the claim needs, but for isopropanol the old mean is
negative, so the ratio (-5.7) cannot express a fold decline and does not reach 51.3; with the
outline grown or shrunk the 3-hexanone ratio is 300 and 24. The measure has an offset: the paraffin
oil trials give a median of -0.23 (per ant -0.45 to 0.075). With each ant's paraffin oil value
subtracted, both declines remain (p = 5.3e-05 and 0.012) and both old means are positive, but the
ratios are 6.0 and 9.5, below 8.1 and 51.3, so the ratios are not reached either way. Without the
minimum filter the map is saturated near the clip value of 4 even for paraffin oil (median 3.80),
since the projection then takes the largest per-pixel noise of 33 planes; that variant measures no
response and is shown only to locate the offset in the minimum filter. The eight other tests (five
odorants at 3%; ethylpyrazine, ethanol and propionic acid at 48%) give p from 0.070 to 0.74, and the
smallest p over the outline, projection and paraffin-subtracted variants is 0.062 (ethylpyrazine at
48%), so the decline is specific to 3-hexanone and isopropanol at 48%. The paper's Figure S3B scale
could not be compared with this offset: its values are not given in the text.

## fig2e-heptanone-breadth-falls, fig2-alarm-sparse, figs3c-general-breadth-no-age-effect

> At 48%, 4-methyl-3-heptanone activates a few more glomeruli in young than in old ants (medians of
> 4 against 2 responding ROIs), while the count for 4-methyl-3-heptanol does not differ with age.

The ROIs were drawn by hand. The claims file's procedure is used: regions of a trial's peak map at
>= 0.2 that recur in at least two trials of any odor, split at glomerulus size (in `antal.candidate_rois`:
local maxima of each map, merged across trials when closer than half a glomerulus, kept when found in
two or more trials, as disks of a glomerulus's diameter). The paper gives no glomerulus size, so
three diameters are run: 25, 35 and 50 pixels (7, 10 and 14 um). An ROI responds to an odor when its
mean peak fold change, averaged over that odor's trials, is >= 0.2 (Methods). Motion-artifact trials:
(a) all trials kept; (b) the trials Data S2 leaves out for that ant (Data S2 holds no ROI counts, so
its trial list is the only record of exclusions). Data S2 holds every trial of every 48% ant for at
least one glomerulus, so a trial missing for every glomerulus excludes nothing; reading (b) therefore
leaves out a trial that Data S2 lacks for one of the ant's glomeruli while it holds that odor's other
repeats there. Wilcoxon rank-sum tests (R's default, normal
approximation with continuity correction when there are ties).

In [ ]:
s2_any = s2[s2.conc == 0.48].groupby("subject").apply(lambda d: set(zip(d.odor_ds, d.rep)), include_groups=False)
missing_every = {s: sorted(set(zip(g.odor, g.rep)) - s2_any.get(s, set()))
                 for s, g in odor_trials[odor_trials.conc == 0.48].groupby("subject")}
print("Trials of 48% ants that Data S2 lacks for every glomerulus:", sum(len(v) for v in missing_every.values()))
# a repeat that Data S2 lacks for a glomerulus while it holds that odor's other repeats (a glomerulus
# without any trial of an odor is a selection of glomeruli, not of trials)
s2_48 = s2[s2.conc == 0.48].drop_duplicates(["subject", "glomerulus", "odor_ds", "rep"])
excluded = {s: sorted({(o, r) for (_, o), x in s2_48[s2_48.subject == s].groupby(["glomerulus", "odor_ds"])
                       for r in (1, 2, 3) if r not in set(x.rep)})
            for s in odor_trials[odor_trials.conc == 0.48].subject.unique()}
print("Trials of 48% ants that Data S2 lacks for a glomerulus while holding the odor's other repeats (reading b excludes them):")
for s, v in excluded.items():
    print(f"  {s}: {v}")
counts = []
for s, g in odor_trials[odor_trials.conc == 0.48].groupby("subject"):
    for reading in ("a", "b"):
        gg = g if reading == "a" else g[[(o, r) not in excluded[s] for o, r in zip(g.odor, g.rep)]]
        mp = [maps[k]["peak"] for k in gg.key]
        for d in (25, 35, 50):
            rois = antal.candidate_rois(mp, masks[s][0], d)
            c = antal.count_responding(mp, list(gg.odor), rois)
            counts.append({"subject": s, "cohort": meta.cohort[s], "reading": reading, "diameter": d,
                           "n_rois": len(rois), **c})
counts = pd.DataFrame(counts)
def wilcox(c, odors):
    rows = []
    for (rd, d), x in c.groupby(["reading", "diameter"]):
        for o in odors:
            y, ol = x[x.cohort == "young"][o], x[x.cohort == "old"][o]
            u, p = stats.mannwhitneyu(y, ol, alternative="two-sided", use_continuity=True)
            rows.append({"reading": rd, "diameter_px": d, "odor": o, "median_young": y.median(),
                         "median_old": ol.median(), "W": u, "p": p})
    return pd.DataFrame(rows)
print("\nCounts of responding ROIs per ant, reading (a), diameter 35 px:")
print(counts[(counts.reading == "a") & (counts.diameter == 35)].drop(columns=["reading", "diameter"]).sort_values("cohort").to_string(index=False))
w2e = wilcox(counts, antal.ALARM)
print("\nAlarm pheromones, young against old:")
print(w2e.to_string(index=False))

**Result: blocked (methods underspecified).** The counts depend on the glomerulus size, which the
paper does not give, and the paper's hand-drawn ROIs and per-ant counts are not published (Data S2
holds no counts, so the claims file's comparison with them cannot be made). At 25 pixels the
medians are 4 (young) and 2 (old) for 4-methyl-3-heptanone, the paper's values, but the Wilcoxon
test gives p = 0.22 (0.17 under reading b); at 35 pixels 1 and 0 (p = 0.35, 0.28 under b); at 50
pixels 0 and 0 (p = 0.71, 0.60 under b). No reading finds the significant difference the paper
reports. For 4-methyl-3-heptanol no reading finds a difference (p 0.23 to 0.95). Data S2 lacks no
trial of a 48% ant for every glomerulus; reading (b) leaves out the three trials it lacks for one
glomerulus while holding that odor's other repeats (one trial in each of three ants).

In [ ]:
mx = counts.groupby(["reading", "diameter"])[antal.ALARM].max()
print("Largest count of responding ROIs over the 14 ants at 48%, per alarm pheromone:")
print(mx.to_string())
print("\nEvery ant at most 25 ROIs (5% of 500) under every reading:", bool((mx <= 25).all().all()),
      "; at most 50 ROIs (10%):", bool((mx <= 50).all().all()))
print("ROIs found per ant (all odors), range:", counts.n_rois.min(), "to", counts.n_rois.max())

**Result: blocked (methods underspecified), as it depends on fig2e-heptanone-breadth-falls.** The
counts are small fractions of 500 under every glomerulus size: the largest count of any ant for
4-methyl-3-heptanone is 32 (25 px), 17 (35 px) and 7 (50 px) ROIs under reading (a) and 39, 23 and
9 under reading (b); for 4-methyl-3-heptanol 11, 4 and 2 under both. The 10% reading holds under
every size; the 5% reading fails at 25 px under both readings (32 and 39 ROIs).

In [ ]:
w3c = wilcox(counts, ["ethylpyrazine", "ethanol", "propionic-acid"])
print(w3c.to_string(index=False))
print("\nThe automated ROIs: ROIs found per ant by diameter (min, median, max):")
print(counts[counts.reading == "a"].groupby("diameter").n_rois.agg(["min", "median", "max"]).to_string())
print("Largest count of responding ROIs per odorant, reading (a):")
print(counts[counts.reading == "a"].groupby("diameter")[antal.GENERAL].max().to_string())


**Result: blocked (methods underspecified).** By the letter of the criterion no odorant shows an
age difference in the count of responding ROIs at any glomerulus size (smallest p 0.075, ethanol at
35 px; ethylpyrazine p 0.083 to 0.32, propionic acid 0.36 to 0.94). But the counts come from the same
automated ROIs on which fig2e-heptanone-breadth-falls is blocked: that procedure does not find the
positive age effect the paper reports for 4-methyl-3-heptanone, and its counts are implausible as
glomerulus counts (17 to 173 ROIs per ant at 25 px, and up to 128 ROIs responding to ethanol, 72 in
one young ant at 35 px, above the general odorants' sparse responses of Figure S3C). A null result
from a procedure that misses the paper's effect does not test the claim, so this claim is blocked
for the same reason as fig2e.

## data-s2-glomerulus-link: finding the focal glomeruli

The paper drew one ROI per focal glomerulus (PGb, PGa, G1 to G4) by hand in each ant and gives no
positions; the dandiset holds none. Data S2 holds, for each ant and glomerulus, the paper's
fold-change trace of every trial it used. Each glomerulus is placed where the dandiset's per-trial
peak fold change (frames 9-14, per plane, in 4 x 4 pixel blocks, averaged over the ROI disk and
maximised over the plane and its neighbours) best correlates (Pearson r over the trials Data S2
holds for that glomerulus) with Data S2's per-trial peak. Only the pattern across trials is matched,
not the amplitude. The AUCs are then read from the dandiset at that place: an ROI disk of radius 8
pixels (2.3 um, a standard choice; 5 and 12 are shown) on the maximum projection of the plane and
its two neighbours, all 40 frames, aligned with the shifts measured on the middle plane of the
trial, fold change from frames 1-5 and the sum over frames 6-40 (Methods), under readings (a) and
(b) of the claims file.

Data S2 holds two quantifications of the same ants: names with a two-digit year (a4-1-18-23) carry
the PGb and PGa alarm traces of Figure 3C and 3D, names with a four-digit year everything else.
One place is found per ant and glomerulus from all of its Data S2 trials; the link is reported for
each set apart. (Locating the Figure 3C and 3D set on its own six trials was tried while preparing, not run
here; with six trials the correlation is near 1 almost anywhere a response is, so it is not used.)

Two checks of the placement. (1) Chance level: the best r over the same voxels when Data S2's values
are shuffled, 200 times per placement, across trials (the verifier's suggestion) and, stricter, by
odor (each odor's three values moved together to another odor), so that a placement is above
chance when its r exceeds the 95th percentile of the shuffled maxima (p < 0.05). (2) Held-out
placement: the place is chosen with the trials whose values are then measured and compared, so the
link, and every test on the located ROIs, is optimistic. Each glomerulus is therefore also placed
once per odor from Data S2's trials of the other odors (leave one odor out), and the AUCs of that
odor's trials are read there; and, for the alarm pheromones, once from the general odorant trials
alone. Placing from the six alarm trials alone, to test the general odorants, is not run: with six
trials the best r is near 1 by chance (the null below shows it for the one glomerulus that Data S2
holds on six trials).

Link criterion (written before the AUCs were compared): an ant, glomerulus and Data S2 set agrees
when the per-trial AUC from the dandiset (reading a, radius 8) correlates with Data S2's at r >= 0.7
across its trials. The link holds when at least 80% of these agree, holds in part from 50%, and
fails below.

In [ ]:
pk = antal.s2_peaks(s2[(s2.conc == 0.48) & s2.subject.isin(meta.index)])
loc_rows, jobs = [], []
tr48 = odor_trials[odor_trials.conc == 0.48].copy()
t0 = time.time()
ctrl = trials[(trials.odor == "paraffin-oil") & (trials.subject.map(meta.conc) == 0.48)]
shift_trials = pd.concat([tr48, ctrl])
SH = dict(zip(shift_trials.key, antal.all_trial_shifts([b[antal.SHIFT_PLANE] for b in shift_trials.blobs])))
print(f"motion over 40 frames measured for {len(SH)} trials in {(time.time() - t0) / 60:.1f} min")
SM = {k: antal.smooth_planes(v["planes_peak"]) for k, v in maps.items() if "planes_peak" in v}
rng_null = np.random.default_rng(773)
held_rows, hjobs = [], []
for (s, g), x in pk.groupby(["subject", "glomerulus"]):
    t2 = tr48[tr48.subject == s].set_index(["odor", "rep"])
    keys = [t2.loc[(o, r), "key"] for o, r in zip(x.odor_ds, x.rep)]
    m = antal._block(masks[s][0].astype(float)) > 0.5
    loc = antal.locate(SM, keys, x.s2_peak.to_numpy(), m, smooth=False)
    n_trial = antal.locate_null(SM, keys, x.s2_peak.to_numpy(), m, n=200, rng=rng_null, smooth=False)
    n_odor = antal.locate_null(SM, keys, x.s2_peak.to_numpy(), m, n=200, rng=rng_null, smooth=False, odors=list(x.odor_ds))
    loc_rows.append({"subject": s, "cohort": meta.cohort[s], "glomerulus": g, "n_trials": len(x), **loc,
                     "null95_trial": np.quantile(n_trial, 0.95), "p_trial": (1 + np.sum(n_trial >= loc["r"])) / 201,
                     "null95_odor": np.quantile(n_odor, 0.95), "p_odor": (1 + np.sum(n_odor >= loc["r"])) / 201})
    for (o, r), row in pd.concat([t2.reset_index(), trials[(trials.subject == s) & (trials.odor == "paraffin-oil")]]).set_index(["odor", "rep"]).iterrows():
        jobs.append(((s, g, o, r), row.blobs, loc["z_index"], loc["y"], loc["x"], 8, (5, 12), SH[row.key], FLOOR[s]))
    # held-out placements: leave one odor out, and for the alarm pheromones the general odorants only
    splits = [(o, (x.odor_ds != o).to_numpy()) for o in x.odor_ds.unique()]
    if x.odor_ds.isin(antal.ALARM).any() and x.odor_ds.isin(antal.GENERAL).any():
        splits.append(("general only", ~x.odor_ds.isin(antal.ALARM).to_numpy()))
    for held, use in splits:
        xx = x[use]
        hk = [t2.loc[(o, r), "key"] for o, r in zip(xx.odor_ds, xx.rep)]
        hl = antal.locate(SM, hk, xx.s2_peak.to_numpy(), m, smooth=False)
        held_rows.append({"subject": s, "glomerulus": g, "held_out": held, "n_place": len(hk), **hl})
        meas = antal.ALARM if held == "general only" else [held]
        for (o, r), row in t2[t2.index.get_level_values(0).isin(meas)].iterrows():
            hjobs.append(((s, g, held, o, r), row.blobs, hl["z_index"], hl["y"], hl["x"], 8, (5, 12), SH[row.key], FLOOR[s]))
locs = pd.DataFrame(loc_rows)
held_locs = pd.DataFrame(held_rows)
print(f"{len(locs)} placements with their chance level and {len(held_locs)} held-out placements in {(time.time() - t0) / 60:.1f} min")
t0 = time.time()
traces = antal.all_roi_traces(jobs)
htraces = antal.all_roi_traces(hjobs)
print(f"{len(traces)} glomerulus traces at the placements and {len(htraces)} at the held-out placements in {(time.time() - t0) / 60:.1f} min")
print("\nPositions found (plane index 0-32, y, x in pixels), the correlation of the per-trial peaks, and its chance level")
print("(95th percentile of the best r with Data S2's values shuffled across trials, or by odor, and the permutation p):")
print(locs.round(3).to_string(index=False))
locs["above_chance"] = locs.p_trial < 0.05
ABOVE = set(zip(locs[locs.above_chance].subject, locs[locs.above_chance].glomerulus))
ABOVE_ODOR = set(zip(locs[locs.p_odor < 0.05].subject, locs[locs.p_odor < 0.05].glomerulus))
print("\nPlacements above chance (p < 0.05), trial shuffle:", locs.groupby("cohort").above_chance.agg(["sum", "count"]).to_dict("index"))
print("Placements above chance (p < 0.05), odor shuffle:", locs.assign(a=locs.p_odor < 0.05).groupby("cohort").a.agg(["sum", "count"]).to_dict("index"))
print("Held-out placements at the same voxel as the placement from all trials:",
      int(held_locs.merge(locs, on=["subject", "glomerulus"], suffixes=("", "_all")).pipe(
          lambda d: ((d.z_index == d.z_index_all) & (d.y == d.y_all) & (d.x == d.x_all)).sum())), "of", len(held_locs))


In [ ]:
def trace_rows(tr, placement):
    rows = []
    for k, v in tr.items():
        s, g, o, r = (k[0], k[1], k[-2], k[-1])
        if len(k) == 5 and k[2] != "general only" and k[2] != o:
            continue
        pl = placement if len(k) == 4 else ("general only" if k[2] == "general only" else "leave one odor out")
        for rad, (A, B) in v.items():
            rows.append({"placement": pl, "subject": s, "glomerulus": g, "odor": o, "rep": r, "radius": rad,
                         "auc_a": A[antal.AUC].sum(), "auc_b": B[antal.AUC].sum(),
                         "pk_a": A[8:14].sum(), "late_a": A[34:40].sum(), "pk_b": B[8:14].sum(), "late_b": B[34:40].sum()})
    return rows
auc = pd.DataFrame(trace_rows(traces, "in sample") + trace_rows(htraces, None))
auc["cohort"] = auc.subject.map(meta.cohort)
print("Trials with no pixel above the F0 floor under reading (b):", int(auc.auc_b.isna().sum()), "of", len(auc))
s2x = s2[(s2.conc == 0.48) & s2.subject.isin(meta.index)]
s2auc = s2x[s2x.frame >= 6].groupby(["subject", "glomerulus", "set", "odor_ds", "rep"]).dff.sum().rename("auc_s2").reset_index()
s2auc = s2auc.rename(columns={"odor_ds": "odor"})
cmp_ = auc[(auc.radius == 8) & (auc.placement == "in sample")].merge(s2auc, on=["subject", "glomerulus", "odor", "rep"])
s2auc = s2auc.drop(columns="set")
trace_r = []
for (s, g, o, r), v in traces.items():
    x = s2x[(s2x.subject == s) & (s2x.glomerulus == g) & (s2x.odor_ds == o) & (s2x.rep == r)].sort_values("frame")
    if len(x) == 40:
        trace_r.append({"subject": s, "glomerulus": g, "set": x.set.iloc[0], "r_trace": np.corrcoef(x.dff, v[8][0])[0, 1]})
trace_r = pd.DataFrame(trace_r)
link = cmp_.groupby(["subject", "cohort", "glomerulus", "set"]).apply(
    lambda d: pd.Series({"n": len(d), "r_auc_a": np.corrcoef(d.auc_s2, d.auc_a)[0, 1],
                         "r_auc_b": np.corrcoef(d.auc_s2, d.auc_b)[0, 1],
                         "median_abs_diff_a": (d.auc_a - d.auc_s2).abs().median()}), include_groups=False).reset_index()
link = link.merge(trace_r.groupby(["subject", "glomerulus", "set"]).r_trace.median().reset_index(), on=["subject", "glomerulus", "set"])
link = link.merge(locs[["subject", "glomerulus", "r", "p_trial"]].rename(columns={"r": "r_locate", "p_trial": "p_locate"}), on=["subject", "glomerulus"])
link["agrees"] = link.r_auc_a >= 0.7
print(link.round(3).to_string(index=False))
frac = link.agrees.mean()
print(f"\nAnt, glomerulus and Data S2 set triples that agree (r >= 0.7): {link.agrees.sum()} of {len(link)} ({frac:.0%})")
print("By glomerulus:", link.groupby("glomerulus").agrees.agg(["sum", "count"]).to_dict("index"))
print("By cohort:", link.groupby("cohort").agrees.agg(["sum", "count"]).to_dict("index"))
print("By Data S2 set:", link.groupby("set").agrees.agg(["sum", "count"]).to_dict("index"))
allr = np.corrcoef(cmp_.auc_s2, cmp_.auc_a)[0, 1]
print(f"Over all {len(cmp_)} trials: r = {allr:.3f}; median dandiset minus Data S2 AUC "
      f"{np.median(cmp_.auc_a - cmp_.auc_s2):.2f} (Data S2 AUC median {cmp_.auc_s2.median():.2f}, IQR "
      f"{cmp_.auc_s2.quantile(.25):.2f} to {cmp_.auc_s2.quantile(.75):.2f})")
print("Link:", "holds" if frac >= 0.8 else "holds in part" if frac >= 0.5 else "fails")
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(cmp_.auc_s2, cmp_.auc_a, s=5); ax[0].set_xlabel("Data S2 AUC"); ax[0].set_ylabel("dandiset AUC (a, r=8)")
lim = [min(cmp_.auc_s2.min(), cmp_.auc_a.min()), max(cmp_.auc_s2.max(), cmp_.auc_a.max())]
ax[0].plot(lim, lim, "k:", lw=0.8)
ax[1].hist(link.r_auc_a, bins=20); ax[1].axvline(0.7, color="k", ls=":"); ax[1].set_xlabel("r per ant and glomerulus")
plt.tight_layout(); plt.show()
print("Axes cover every point; ranges", [round(v, 1) for v in lim])

# held-out check: the same criterion on AUCs read where the glomerulus was placed without that odor
cmp_ho = auc[(auc.radius == 8) & (auc.placement == "leave one odor out")].merge(
    cmp_[["subject", "glomerulus", "odor", "rep", "set", "auc_s2"]], on=["subject", "glomerulus", "odor", "rep"])
link_ho = cmp_ho.groupby(["subject", "cohort", "glomerulus", "set"]).apply(
    lambda d: pd.Series({"n": len(d), "r_auc_a": np.corrcoef(d.auc_s2, d.auc_a)[0, 1]}), include_groups=False).reset_index()
link_ho["agrees"] = link_ho.r_auc_a >= 0.7
print(f"\nHeld out (leave one odor out): {link_ho.agrees.sum()} of {len(link_ho)} triples agree ({link_ho.agrees.mean():.0%}); "
      f"by cohort {link_ho.groupby('cohort').agrees.agg(['sum', 'count']).to_dict('index')}; "
      f"by set {link_ho.groupby('set').agrees.agg(['sum', 'count']).to_dict('index')}; "
      f"over all {len(cmp_ho)} trials r = {np.corrcoef(cmp_ho.auc_s2, cmp_ho.auc_a)[0, 1]:.3f}")
cmp_gen = auc[(auc.radius == 8) & (auc.placement == "general only")].merge(
    cmp_[["subject", "glomerulus", "odor", "rep", "set", "auc_s2"]], on=["subject", "glomerulus", "odor", "rep"])
link_gen = cmp_gen.groupby(["subject", "cohort", "glomerulus", "set"]).apply(
    lambda d: pd.Series({"n": len(d), "r_auc_a": np.corrcoef(d.auc_s2, d.auc_a)[0, 1]}), include_groups=False).reset_index()
link_gen["agrees"] = link_gen.r_auc_a >= 0.7
print(f"Alarm trials, placed from the general odorants only: {link_gen.agrees.sum()} of {len(link_gen)} agree; "
      f"by cohort {link_gen.groupby('cohort').agrees.agg(['sum', 'count']).to_dict('index')}")
for name, lk in (("in sample (criterion)", link), ("leave one odor out", link_ho)):
    lk = lk.assign(above=[(s, g) in ABOVE for s, g in zip(lk.subject, lk.glomerulus)])
    print(f"{name}: triples that agree among placements above chance {lk[lk.above].agrees.sum()} of {lk.above.sum()}, "
          f"below chance {lk[~lk.above].agrees.sum()} of {(~lk.above).sum()}")


**Result: the link holds in part (partially reproduced).** The 14 ants of Data S2 at 48% are the 14
dandiset ants at 48%, with matching ages. At the place found for each glomerulus, 71 of 108 ant,
glomerulus and set triples (66%) have per-trial AUCs that correlate with Data S2's at r >= 0.7,
below the 80% the criterion asks. Young ants agree far more often (46 of 56) than old ants (25 of
52), and the main set (57 of 81) more often than the Figure 3C and 3D set of PGb and PGa alarm
traces (14 of 27). Over all 1672 trials r = 0.793.

The two checks show that this 66% is optimistic. Every young placement is above chance (42 of 42),
but only 24 of 39 old placements are under the trial shuffle and 18 of 39 under the odor shuffle;
the triples agree at 66 of 90 placements above chance and 5 of 18 below. Read where the glomerulus
was placed without the odor's own trials, 44 of 108 triples agree (41%; young 35 of 56, old 9 of
52; Figure 3C and 3D set 4 of 27), over all trials r = 0.731; the alarm trials read where the
glomerulus was placed from the general odorants alone agree in 19 of 79 (old 4 of 37). The
criterion, written for the placement from all trials, gives a partial link; on held-out trials
the link would fail, and in old ants it fails either way.

In [ ]:
# stimulus timing from the glomerulus traces: mean fold change per frame, odor trials with a strong
# response (Data S2 AUC in the top quarter) against the paraffin oil trials
strong = set(map(tuple, cmp_[cmp_.auc_s2 >= cmp_.auc_s2.quantile(0.75)][["subject", "glomerulus", "odor", "rep"]].to_numpy()))
tc_odor = np.mean([v[8][0] for k, v in traces.items() if k in strong], 0)
tc_par = np.mean([v[8][0] for k, v in traces.items() if k[2] == "paraffin-oil"], 0)
tc = pd.DataFrame({"frame": np.arange(1, 41), "strong odor trials": tc_odor, "paraffin oil": tc_par})
print(f"{len(strong)} strong odor trials, {sum(k[2] == 'paraffin-oil' for k in traces)} paraffin oil traces")
print(tc.round(3).T.to_string(header=False))
first_up = int(np.argmax(tc_odor > tc_odor[:5].mean() + 3 * tc_odor[:5].std())) + 1
print("First frame above the frames 1-5 mean + 3 SD (strong odor trials):", first_up,
      "; peak frame:", int(np.argmax(tc_odor)) + 1)
print("Mean of frames 9-14: strong odor trials", round(tc_odor[8:14].mean(), 3), "; paraffin oil", round(tc_par[8:14].mean(), 3))
per_trial = pd.DataFrame([{"kind": "paraffin oil" if k[2] == "paraffin-oil" else "odor, Data S2 AUC top quarter" if k in strong else "odor, other",
                           "peak_minus_base": v[8][0][8:14].mean() - v[8][0][:5].mean(),
                           "frames_6_8_minus_base": v[8][0][5:8].mean() - v[8][0][:5].mean()} for k, v in traces.items()])
print("\nPer trace, frames 9-14 (and 6-8) minus frames 1-5, by kind of trial:")
print(per_trial.groupby("kind").agg(["count", "median"]).round(3).to_string())

## The focal glomerulus claims

Per ant, glomerulus and odor, the mean AUC over the trials Data S2 holds for it (the paper's own
selection of trials; all three trials as a variant), at 48%. Only ants that have the glomerulus in
Data S2 enter (the Figure S2 and S4 n). The main measure is read at the held-out placement (leave one
odor out: each odor's AUCs are read where the glomerulus was placed from the other odors' trials), so
that the values tested are not those the place was chosen to match; this is an assumption (the
paper drew the ROIs by hand). Reading (a), radius 8. The variants: reading (b), radii 5 and 12, all
three trials, placements above chance only, the placement from all trials (in sample, the main
measure of round 1), and for the alarm pheromones the placement from the general odorants only. As
a reference only, the same test is run on Data S2's own traces.

In [ ]:
MAIN = ("leave one odor out", "a", 8, "s2", False)    # placement, reading, radius, trials, above chance only
VARIANTS = [MAIN, ("leave one odor out", "b", 8, "s2", False), ("leave one odor out", "a", 5, "s2", False),
            ("leave one odor out", "a", 12, "s2", False), ("leave one odor out", "a", 8, "all", False),
            ("leave one odor out", "a", 8, "s2", True), ("in sample", "a", 8, "s2", False),
            ("in sample", "b", 8, "s2", False), ("in sample", "a", 8, "s2", True)]
ALARM_VARIANTS = VARIANTS + [("general only", "a", 8, "s2", False), ("general only", "b", 8, "s2", False)]
def vname(v):
    pl, rd, rad, tf, ab = v
    return f"{pl}, {rd}, r={rad}" + (", all trials" if tf == "all" else "") + (", above chance" if ab else "")
def variants_for(odors):
    return ALARM_VARIANTS if set(odors) <= set(antal.ALARM) else VARIANTS
def per_ant_auc(odors, v=MAIN, col="auc"):
    pl, rd, rad, tf, ab = v
    a = auc[(auc.placement == pl) & (auc.radius == rad) & auc.odor.isin(odors)]
    if tf == "s2":
        a = a.merge(s2auc[["subject", "glomerulus", "odor", "rep"]], on=["subject", "glomerulus", "odor", "rep"])
    else:
        a = a[a.set_index(["subject", "glomerulus"]).index.isin(s2auc.set_index(["subject", "glomerulus"]).index.unique())]
    if ab:
        a = a[[(s, g) in ABOVE for s, g in zip(a.subject, a.glomerulus)]]
    return a.groupby(["subject", "cohort", "glomerulus", "odor"])[f"{col}_{rd}"].mean().rename("v").reset_index()
def s2_per_ant(odors, s2df=None):
    d = s2auc if s2df is None else s2df
    d = d[d.odor.isin(odors)].copy()
    d["cohort"] = np.where(d.subject.map(meta.age_days) < 40, "young", "old")
    return d.groupby(["subject", "cohort", "glomerulus", "odor"]).auc_s2.mean().rename("v").reset_index()
def welch_glom(pa_, pairs):
    rows = []
    for g, o in pairs:
        d = pa_[(pa_.glomerulus == g) & (pa_.odor == o)]
        y, ol = d[d.cohort == "young"].v, d[d.cohort == "old"].v
        t, p = stats.ttest_ind(y, ol, equal_var=False) if len(y) > 1 and len(ol) > 1 else (np.nan, np.nan)
        rows.append({"glomerulus": g, "odor": o, "n_young": len(y), "n_old": len(ol), "mean_young": y.mean(),
                     "mean_old": ol.mean(), "t": t, "p": p})
    return pd.DataFrame(rows)
def run_all(pairs, odors):
    out = {vname(v): welch_glom(per_ant_auc(odors, v), pairs) for v in variants_for(odors)}
    out["Data S2 (reference)"] = welch_glom(s2_per_ant(odors), pairs)
    return out
def show_all(out):
    t = pd.concat([v.assign(variant=k) for k, v in out.items()])
    t = t.sort_values(["glomerulus", "odor"], kind="stable")
    with pd.option_context("display.float_format", "{:.4g}".format):
        print(t[["glomerulus", "odor", "variant", "n_young", "n_old", "mean_young", "mean_old", "t", "p"]].to_string(index=False))
GLOMS = ["PGb", "PGa", "G1", "G2", "G3", "G4"]
MAIN_NAME = vname(MAIN)

### figs2a-mean-six-glomeruli-no-age-effect

> The alarm pheromone response averaged over the six focal glomeruli does not differ between young
> and old ants.

In [ ]:
def mean_over_gloms(pa_):
    return pa_.groupby(["subject", "cohort", "odor"]).v.mean().reset_index().assign(glomerulus="mean of six")
res_s2a = {vname(v): welch_glom(mean_over_gloms(per_ant_auc(antal.ALARM, v)), [("mean of six", o) for o in antal.ALARM])
           for v in ALARM_VARIANTS}
res_s2a["Data S2 (reference)"] = welch_glom(mean_over_gloms(s2_per_ant(antal.ALARM)), [("mean of six", o) for o in antal.ALARM])
show_all(res_s2a)

**Result: reproduced (resting on the located ROIs).** The mean AUC over the six glomeruli does not
differ with age: at the held-out placement (reading a, radius 8) p = 0.58 (4-methyl-3-heptanone) and
0.23 (4-methyl-3-heptanol); over every variant, including the placement from all trials of round 1
(p = 0.81 and 0.25) and placements above chance only, p is 0.0799 or more (general odorants only,
reading b, 4-methyl-3-heptanone). Data S2's own values give p = 0.14 and 0.053. This null result is
weak evidence: it comes from the same located ROIs that miss the paper's positive old-ant effects
(fig3b, fig3d), and a null from a procedure that misses the paper's effects is a weak test, the
reason figs3c-general-breadth-no-age-effect is blocked. It is kept as reproduced because the six
glomeruli are placed from Data S2 per ant (above chance in every young ant), not found by the
automated rule of figs3c, and no variant reaches p < 0.05.

### fig3b-g2-heptanone-weaker-old

> In glomeruli G1 to G4 alarm pheromone responses tend to weaken with age; the decline is
> significant only for 4-methyl-3-heptanone in G2.

In [ ]:
res_3b = run_all([(g, o) for g in ["G1", "G2", "G3", "G4"] for o in antal.ALARM], antal.ALARM)
show_all(res_3b)
for k, v in res_3b.items():
    g2 = v[(v.glomerulus == "G2") & (v.odor == "4-methyl-3-heptanone")].iloc[0]
    others = v[~((v.glomerulus == "G2") & (v.odor == "4-methyl-3-heptanone"))]
    print(f"{k:>24}: G2 heptanone p = {g2.p:.3g}, young higher {g2.mean_young > g2.mean_old}; "
          f"other seven with p < 0.05: {int((others.p < 0.05).sum())}; young above old in {int((v.mean_young > v.mean_old).sum())} of 8")

**Result: not reproduced.** At the held-out placement (reading a, radius 8) the G2 response to
4-methyl-3-heptanone does not differ with age (young -1.98, old -2.34, p = 0.86), and G1 with
4-methyl-3-heptanone does (8.71 against -1.97, p = 0.033). Over the variants the G2 p ranges from
0.0496 to 0.86; only the placement from all trials restricted to placements above chance (5 old
ants) gives G2 p < 0.05, with one other pair also significant, which would be a partial
reproduction. Young means are above old ones in 5 of 8 pairs (held out). Data S2's own values give G2
p = 0.0069 and no other pair below 0.05, as the paper.

### fig3d-pg-stronger-old

> PGb and PGa respond significantly more strongly to alarm pheromones in old than in young ants; PGa
> does not respond to 4-methyl-3-heptanone.

In [ ]:
res_3d = run_all([(g, o) for g in ["PGb", "PGa"] for o in antal.ALARM], antal.ALARM)
show_all(res_3d)
def sig_old(v, g, o):
    r = v[(v.glomerulus == g) & (v.odor == o)].iloc[0]
    return bool(r.p < 0.05 and r.mean_old > r.mean_young)
for k, v in res_3d.items():
    ra = sig_old(v, "PGb", antal.ALARM[0]) and sig_old(v, "PGb", antal.ALARM[1]) and sig_old(v, "PGa", antal.ALARM[1])
    rb = (sig_old(v, "PGb", antal.ALARM[0]) or sig_old(v, "PGb", antal.ALARM[1])) and sig_old(v, "PGa", antal.ALARM[1])
    print(f"{k:>24}: reading (a) holds {ra}; reading (b) holds {rb}")
pa_d = per_ant_auc(antal.ALARM).rename(columns={"v": "dandiset, held out"})
pa_i = per_ant_auc(antal.ALARM, ("in sample", "a", 8, "s2", False)).rename(columns={"v": "dandiset, in sample"})
pa_s = s2_per_ant(antal.ALARM).rename(columns={"v": "Data S2"})
x = pa_d.merge(pa_i, on=["subject", "cohort", "glomerulus", "odor"]).merge(pa_s, on=["subject", "cohort", "glomerulus", "odor"])
x["above_chance"] = [(s, g) in ABOVE for s, g in zip(x.subject, x.glomerulus)]
print("\nPer ant mean AUC (a, r=8), PGb and PGa, dandiset at the held-out and in-sample placements against Data S2:")
print(x[x.glomerulus.isin(["PGb", "PGa"])].sort_values(["glomerulus", "odor", "cohort"]).round(2).to_string(index=False))

**Result: not reproduced.** At the held-out placement (reading a, radius 8) old ants' PGb and PGa
alarm AUCs are higher on average but no difference is significant: PGb with 4-methyl-3-heptanone
-0.21 against 8.46 (p = 0.14), PGb with 4-methyl-3-heptanol p = 0.43, PGa with 4-methyl-3-heptanol
-4.21 against 0.08 (p = 0.24). No variant meets either reading. The one pair significant at the
placement from all trials, PGa with 4-methyl-3-heptanol (11.6 against -1.35, p = 0.044), is not
significant at any held-out placement (p 0.13 to 0.33), so it came from choosing the place on the
same trials. With the placement from the general odorants alone, PGb with 4-methyl-3-heptanone is
higher in old ants (p = 0.026) but PGa is not (p = 0.33). PGa's 4-methyl-3-heptanone AUC is -3.32
(young) and 1.38 (old), no response. Data S2's own values meet both readings (PGb p = 0.036 and
0.0036, PGa 0.016). At the held-out placement the old ants' PGb AUCs are below Data S2's in 6 of 7 ants for
4-methyl-3-heptanone and 7 of 7 for 4-methyl-3-heptanol (per-ant table).

### fig3d-pg-60-vs-90-days-same

> PGb and PGa responses to alarm pheromones do not differ between 60-day-old ants and the 90-day-old
> ants of the earlier study.

Reading (b) of the claims file: the 90-day per-ant values are those of Data S2 (its 2022 ants at
48%, AUC frames 6-40 from its traces, mean over trials); the 60-day values are the dandiset's, so
this reading compares values measured here with values measured by the paper. Reading (a) follows.

In [ ]:
s2_90 = s2[(s2.age == "3months") & (s2.conc == 0.48) & s2.glomerulus.isin(["PGb", "PGa"]) & s2.odor_ds.isin(antal.ALARM)]
a90 = (s2_90[s2_90.frame >= 6].groupby(["subject", "glomerulus", "odor_ds", "trial"]).dff.sum()
       .groupby(["subject", "glomerulus", "odor_ds"]).mean().rename("v").reset_index().rename(columns={"odor_ds": "odor"}))
print("90-day ants in Data S2 at 48% per glomerulus and odor:")
print(a90.groupby(["glomerulus", "odor"]).subject.nunique().to_string())
rows = []
for name, pa60 in [(vname(v), per_ant_auc(antal.ALARM, v)) for v in ALARM_VARIANTS] + [("Data S2 60-day (reference)", s2_per_ant(antal.ALARM))]:
    for g, o in [("PGb", antal.ALARM[0]), ("PGb", antal.ALARM[1]), ("PGa", antal.ALARM[1])]:
        x60 = pa60[(pa60.cohort == "old") & (pa60.glomerulus == g) & (pa60.odor == o)].v
        x90 = a90[(a90.glomerulus == g) & (a90.odor == o)].v
        t, p = stats.ttest_ind(x60, x90, equal_var=False)
        rows.append({"60-day values": name, "glomerulus": g, "odor": o, "n60": len(x60), "n90": len(x90),
                     "mean60": x60.mean(), "mean90": x90.mean(), "t": t, "p": p})
r6090 = pd.DataFrame(rows)
print(r6090.to_string(index=False))

Reading (a): the 90-day AUCs recomputed from DANDI 000467 (version 0.261004.1655, the latest
published), the earlier study's dandiset, which has the same file layout. The eight 90-day ants of
Data S2 at 48% are located in it as above (PGb from its 12 Data S2 trials with four odors, PGa from
its 6 alarm trials), and their AUCs are measured as for the 60-day ants.

In [ ]:
OLD_ID, OLD_VERSION = antal.OLD_DANDISET
a467 = antal.list_assets(OLD_ID, OLD_VERSION)
t467 = antal.trial_table(a467)
s2_90all = s2[(s2.age == "3months") & (s2.conc == 0.48)]
subs90 = sorted(s2_90all.subject.unique())
t90 = t467[t467.subject.isin(subs90) & (t467.conc == 0.48)].copy()
print("Trials of these ants at 48% by number of frames (the 120-frame trials are left out):",
      t90.groupby(["odor", "frames"]).t.count().to_dict())
t90 = t90[t90.frames == 40]
t90["key"] = list(zip(t90.subject, t90.t))
print(f"DANDI {OLD_ID} version {OLD_VERSION}: {len(a467)} assets, {t467.subject.nunique()} ants; "
      f"the {len(subs90)} Data S2 90-day ants at 48% are all in it: {set(subs90) <= set(t467.subject)}")
print(t90.groupby(["subject", "odor"]).t.count().unstack(fill_value=0).to_string())
with ThreadPoolExecutor(8) as ex:
    m90 = dict(zip(subs90, ex.map(lambda s: antal.nwb_metadata(t90[t90.subject == s].blobs.iloc[0][16]), subs90)))
print("Ages (NWB):", {s: m["age"] for s, m in m90.items()}, "; offsets:", {m["offset"] for m in m90.values()},
      "; shapes:", {m["shape"] for m in m90.values()})
print(antal.check_offsets(t90.sample(2, random_state=0).blobs.map(lambda b: b[3]).tolist()).to_string(index=False))
t0 = time.time()
maps90 = antal.all_volume_maps(dict(zip(t90.key, t90.blobs)), keep_keys=list(t90.key), processes=4)
SH90 = dict(zip(t90.key, antal.all_trial_shifts([b[antal.SHIFT_PLANE] for b in t90.blobs])))
pk90 = antal.s2_peaks(s2_90all)
chk = s2_90all.dropna(subset=["t_s2"]).drop_duplicates(["subject", "odor_ds", "rep", "t_s2"])
chk = chk.merge(t467[t467.conc == 0.48][["subject", "odor", "rep", "t"]], left_on=["subject", "odor_ds", "rep"], right_on=["subject", "odor", "rep"], how="left")
print(f"Data S2 rows naming the trial number: {len(chk)}; the dandiset's trial of that odor and repeat has that number: {int((chk.t == chk.t_s2).sum())}")
loc90, jobs90 = [], []
rng90 = np.random.default_rng(467)
FLOOR90 = {}
for (s, g), x in pk90.groupby(["subject", "glomerulus"]):
    tt_ = t90[t90.subject == s].set_index(["odor", "rep"])
    raw = np.mean([maps90[k]["raw_max"] for k in tt_.key], 0)
    al90 = antal.al_mask(raw)
    FLOOR90[s] = float(np.median(raw[~al90]))
    m = antal._block(al90.astype(float)) > 0.5
    x = x[[(o, r) in tt_.index for o, r in zip(x.odor_ds, x.rep)]]
    keys = [tt_.loc[(o, r), "key"] for o, r in zip(x.odor_ds, x.rep)]
    pl90 = {k: maps90[k]["planes_peak"] for k in tt_.key}
    loc = antal.locate(pl90, keys, x.s2_peak.to_numpy(), m)
    nul = antal.locate_null(pl90, keys, x.s2_peak.to_numpy(), m, n=200, rng=rng90)
    loc90.append({"subject": s, "glomerulus": g, "n_trials": len(x), **loc, "null95_trial": np.quantile(nul, 0.95),
                  "p_trial": (1 + np.sum(nul >= loc["r"])) / 201})
    for (o, r), row in tt_.iterrows():
        jobs90.append(((s, g, o, r), row.blobs, loc["z_index"], loc["y"], loc["x"], 8, (5, 12), SH90[row.key], FLOOR90[s]))
tr90 = antal.all_roi_traces(jobs90)
print(f"{len(maps90)} trials and {len(tr90)} glomerulus traces in {(time.time() - t0) / 60:.1f} min")
auc90 = pd.DataFrame([{"subject": s, "glomerulus": g, "odor": o, "rep": r, "radius": rad,
                       "auc_a": A[antal.AUC].sum(), "auc_b": B[antal.AUC].sum()}
                      for (s, g, o, r), v in tr90.items() for rad, (A, B) in v.items()])
s2auc90 = (s2_90all[s2_90all.frame >= 6].groupby(["subject", "glomerulus", "odor_ds", "rep"]).dff.sum()
           .rename("auc_s2").reset_index().rename(columns={"odor_ds": "odor"}))
c90 = auc90[auc90.radius == 8].merge(s2auc90, on=["subject", "glomerulus", "odor", "rep"])
l90 = c90.groupby(["subject", "glomerulus"]).apply(lambda d: pd.Series({"n": len(d), "r_auc_a": np.corrcoef(d.auc_s2, d.auc_a)[0, 1]}),
                                                   include_groups=False).reset_index()
l90 = l90.merge(pd.DataFrame(loc90)[["subject", "glomerulus", "z_index", "y", "x", "r", "n_trials", "null95_trial", "p_trial"]], on=["subject", "glomerulus"])
print("\n90-day ants: positions found and agreement of per-trial AUCs with Data S2 (r >= 0.7):")
print(l90.to_string(index=False))
print("Agree:", int((l90.r_auc_a >= 0.7).sum()), "of", len(l90), "; placements above chance (p < 0.05):",
      l90.assign(a=l90.p_trial < 0.05).groupby("glomerulus").a.agg(["sum", "count"]).to_dict("index"))
rows = []
# the 60-day values at the placement from all trials (in sample) and, added after the round 2
# verifier, at the held-out placements of the main measure; the 90-day values are always in sample
for pl60, rd, rad in [("in sample", rd, rad) for rd, rad in (("a", 8), ("b", 8), ("a", 5), ("a", 12), ("b", 5), ("b", 12))] \
        + [(pl, rd, 8) for pl in ("leave one odor out", "general only") for rd in ("a", "b")]:
    pa60 = per_ant_auc(antal.ALARM, (pl60, rd, rad, "s2", False))
    a90r = (auc90[auc90.radius == rad].merge(s2auc90[["subject", "glomerulus", "odor", "rep"]], on=["subject", "glomerulus", "odor", "rep"])
            .groupby(["subject", "glomerulus", "odor"])[f"auc_{rd}"].mean().rename("v").reset_index())
    for g, o in [("PGb", antal.ALARM[0]), ("PGb", antal.ALARM[1]), ("PGa", antal.ALARM[1])]:
        x60 = pa60[(pa60.cohort == "old") & (pa60.glomerulus == g) & (pa60.odor == o)].v
        x90 = a90r[(a90r.glomerulus == g) & (a90r.odor == o)].v
        t, p = stats.ttest_ind(x60, x90, equal_var=False)
        rows.append({"60-day placement": pl60, "variant": f"{rd}, r={rad}", "glomerulus": g, "odor": o, "n60": len(x60), "n90": len(x90),
                     "mean60": x60.mean(), "mean90 (000467)": x90.mean(), "t": t, "p": p})
r6090a = pd.DataFrame(rows)
print("\nReading (a): 60-day (dandiset 000773) against 90-day (DANDI 000467), both measured here:")
print(r6090a.to_string(index=False))

**Result: blocked (methods underspecified).** The two readings give different outcomes and neither
can be run as a fair test here. Reading (a), both cohorts measured here at the placement from all
trials, finds no difference (PGb with 4-methyl-3-heptanone 5.15 against 11.59, p = 0.15; PGb with
4-methyl-3-heptanol p = 0.94; PGa p = 0.41). With the 60-day values read at the held-out placement
of the main measure (90-day values still from the placement from all trials), it also finds none
(8.46 against 11.59, p = 0.60; p = 0.53; PGa 0.08 against 6.53, p = 0.18; reading b p = 0.33 to
0.74); at the placement from the general odorants alone only reading (b) of PGb with
4-methyl-3-heptanone differs (1.74 against 14.14, p = 0.041). But the 90-day PGa positions are placed from six
trials, and only 1 of 8 is above chance (r 0.996 to 0.9997 against 95th percentiles of 0.999); the
PGb positions (12 trials) are above chance in 7 of 8. A held-out placement cannot be made for the
90-day PGa, whose Data S2 trials are the six alarm trials. Reading (b) compares the 60-day AUCs
measured here with Data S2's 90-day values; it finds differences (held-out placement p = 0.014,
0.022 and 0.016; placement from all trials p = 0.0002, 0.053, 0.41), but the AUCs measured here are
several times smaller than Data S2's for the same ants (60-day PGb with 4-methyl-3-heptanone 5.15
against Data S2's 20.66 for the same ants; 90-day 11.59 in DANDI 000467 against 24.99), so the
difference reflects the two ways of measuring, not age. With Data S2's values for both ages, the only
comparison within one pipeline (it uses no dandiset data), none differs (p = 0.44, 0.80 and 0.78).
In 000467, 12 of 16 ant and glomerulus pairs agree with Data S2 at r >= 0.7, and one of 96 Data S2
trial numbers does not match the dandiset's trial of that odor and repeat.

### figs4a-general-odorants-in-focal-glomeruli

> 3-hexanone excites all six focal glomeruli in young ants and PGb, PGa and G4 in old ants;
> isopropanol excites all but G1 in young ants only; propionic acid excites G4 at both ages; no
> other general odorant excites a focal glomerulus.

One-sided one-sample t test that the mean AUC exceeds zero, per cohort, glomerulus and odorant (60
cells), at p < 0.05 uncorrected.

In [ ]:
EXPECTED_S4A = {("young", g, "3-hexanone") for g in GLOMS} | {("young", g, "isopropanol") for g in GLOMS if g != "G1"} \
    | {("young", "G4", "propionic-acid"), ("old", "PGb", "3-hexanone"), ("old", "PGa", "3-hexanone"),
       ("old", "G4", "3-hexanone"), ("old", "G4", "propionic-acid")}
def one_sample(pa_):
    rows = []
    for co in ("young", "old"):
        for g in GLOMS:
            for o in antal.GENERAL:
                v = pa_[(pa_.cohort == co) & (pa_.glomerulus == g) & (pa_.odor == o)].v
                t, p = stats.ttest_1samp(v, 0, alternative="greater") if len(v) > 1 else (np.nan, np.nan)
                rows.append({"cohort": co, "glomerulus": g, "odor": o, "n": len(v), "mean": v.mean(), "p": p,
                             "significant": bool(p < 0.05), "expected": (co, g, o) in EXPECTED_S4A})
    return pd.DataFrame(rows)
res_s4a = {vname(v): one_sample(per_ant_auc(antal.GENERAL, v)) for v in VARIANTS}
res_s4a["Data S2 (reference)"] = one_sample(s2_per_ant(antal.GENERAL))
main = res_s4a[MAIN_NAME]
print(main.to_string(index=False))
def s4a_summary(v):
    hx = v[v.odor.isin(["3-hexanone", "isopropanol"])]
    return {"matches": int((v.significant == v.expected).sum()), "missing": int((~v.significant & v.expected).sum()),
            "extra": int((v.significant & ~v.expected).sum()),
            "hexanone_isopropanol_pattern_holds": bool((hx.significant == hx.expected).all())}
print()
print(pd.DataFrame({k: s4a_summary(v) for k, v in res_s4a.items()}).T.to_string())
for k in (MAIN_NAME, vname(VARIANTS[1]), "in sample, a, r=8"):
    v = res_s4a[k]
    print(f"\n{k}: mismatches", v[v.significant != v.expected][["cohort", "glomerulus", "odor", "mean", "p", "expected"]].to_string(index=False))

**Result: not reproduced (resting on the located ROIs and ROI size).** At the held-out placement
(reading a, radius 8) 56 of the 60 cells agree with the figure and no unexpected cell is
significant. The four that differ are young G1 with 3-hexanone (p = 0.058), old G4 with
3-hexanone (p = 0.15) and G4 with propionic acid at both ages (p = 0.44 and 0.39). The misses for
3-hexanone break the text's pattern, so the claim is not reproduced. The pattern holds under one of
the nine variants, the placement from all trials restricted to placements above chance (58 of 60),
which would be a partial reproduction; at the placement from all trials it fails on old G4 with
3-hexanone (p = 0.063), as in round 1. Data S2's own values give all 60.

### fig4-general-odorants-decline

> Responses to 3-hexanone and isopropanol decline significantly with age in five of the six focal
> glomeruli (3-hexanone in all but PGb, isopropanol in all but G1).

In [ ]:
res_4 = run_all([(g, o) for o in ["3-hexanone", "isopropanol"] for g in GLOMS], ["3-hexanone", "isopropanol"])
show_all(res_4)
EXP4 = {(g, "3-hexanone") for g in GLOMS if g != "PGb"} | {(g, "isopropanol") for g in GLOMS if g != "G1"}
def s4_summary(v):
    dec = (v.p < 0.05) & (v.mean_young > v.mean_old)
    e = np.array([(g, o) in EXP4 for g, o in zip(v.glomerulus, v.odor)])
    per_odor = {o: int(dec[(v.odor == o).to_numpy() & e].sum()) for o in ["3-hexanone", "isopropanol"]}
    return {"expected declines found (of 10)": int(dec[e].sum()), "3-hexanone (of 5)": per_odor["3-hexanone"],
            "isopropanol (of 5)": per_odor["isopropanol"],
            "PGb 3-hexanone p": float(v[(v.glomerulus == "PGb") & (v.odor == "3-hexanone")].p.iloc[0]),
            "G1 isopropanol p": float(v[(v.glomerulus == "G1") & (v.odor == "isopropanol")].p.iloc[0])}
print()
print(pd.DataFrame({k: s4_summary(v) for k, v in res_4.items()}).T.round(4).to_string())

**Result: partially reproduced (resting on the held-out placement and the ROI size).** At the
held-out placement (reading a, radius 8) 9 of the 10 declines are significant: 3-hexanone in PGa,
G2, G3 and G4 (p = 0.016, 0.0498, 0.048, 0.020) but not G1 (p = 0.12), and isopropanol in all five
(PGb 0.008, PGa 0.0098, G2 0.019, G3 0.021, G4 0.025), so at least four of five glomeruli decline
for each odorant. 3-hexanone in PGb, not significant in the paper, declines here (29.0 against
12.8, p = 0.022); isopropanol in G1 does not (p = 0.20), as in the paper. The partial level rests
on two p-values just under 0.05 and is not met by any of the eight other variants (5 to 8 of 10;
3-hexanone or isopropanol 3 or fewer of 5), including the placement from all trials (7 of 10,
isopropanol 3 of 5). Data S2's own values give all 10, with PGb 3-hexanone p = 0.23. The
located ROIs bias this claim toward passing: on held-out trials only 9 of 52 old-ant glomerulus
traces follow Data S2's, so old-ant values are read partly from places that do not respond like the
glomerulus, which pulls old means down. Claims that need old ants to be lower (this one) are pushed
toward passing, and those that need them higher (fig3d, the late window of figs4d) toward failing.
The PGb decline with 3-hexanone is the visible case: the old mean is 12.8 here against Data S2's 25.1
for the same ants (p = 0.022 here, 0.23 on Data S2). With all trials in place of Data S2's trials,
G2 with 3-hexanone gives p = 0.059.

### fig4-propionic-acid-g2-g3

> In young ants propionic acid gives a very small negative deflection in G2 and G3 that declines in
> amplitude with age; its responses in G4 tend to fall with age, not significantly.

In [ ]:
res_pa = run_all([(g, "propionic-acid") for g in ["G2", "G3", "G4"]], ["propionic-acid"])
show_all(res_pa)
for k, v in res_pa.items():
    g2, g3, g4 = (v[v.glomerulus == g].iloc[0] for g in ("G2", "G3", "G4"))
    ok = (g2.mean_young < 0 and g3.mean_young < 0 and g2.p < 0.05 and g3.p < 0.05 and abs(g2.mean_old) < abs(g2.mean_young)
          and abs(g3.mean_old) < abs(g3.mean_young) and g4.p >= 0.05 and g4.mean_old < g4.mean_young)
    print(f"{k:>24}: criterion holds {ok}")

**Result: not reproduced.** At the held-out placement (reading a, radius 8) the propionic acid AUC
of young ants is negative in G2 (-5.1) and G3 (-4.9) and nearer zero in old ants (-3.3 and -0.83),
the stated direction, but neither difference is significant (p = 0.60 and 0.17); no variant meets the
criterion. G4 does not differ (0.92 against 0.75, p = 0.98). Data S2's own values give G2 p = 0.0050
and G3 p = 0.030.

### figs4d-pgb-hexanone-dynamics

> In PGb, 3-hexanone responses of old ants have a lower peak but decline more slowly than those of
> young ants; the AUC over frames 9 to 14 is larger in young ants and over frames 35 to 40 larger
> in old ants.

In [ ]:
rows = []
for v in VARIANTS:
    for win in ("pk", "late"):
        pa_ = per_ant_auc(["3-hexanone"], v, col=win)
        rows.append({"variant": vname(v), "window": "frames 9-14" if win == "pk" else "frames 35-40",
                     **welch_glom(pa_, [("PGb", "3-hexanone")]).iloc[0].drop(["glomerulus", "odor"]).to_dict()})
s2p = s2x[(s2x.glomerulus == "PGb") & (s2x.odor_ds == "3-hexanone")]
for win, (f0, f1) in (("frames 9-14", (9, 14)), ("frames 35-40", (35, 40))):
    d = s2p[(s2p.frame >= f0) & (s2p.frame <= f1)].groupby(["subject", "trial"]).dff.sum().groupby("subject").mean().rename("v").reset_index()
    d["cohort"] = np.where(d.subject.map(meta.age_days) < 40, "young", "old"); d["glomerulus"] = "PGb"; d["odor"] = "3-hexanone"
    rows.append({"variant": "Data S2 (reference)", "window": win, **welch_glom(d, [("PGb", "3-hexanone")]).iloc[0].drop(["glomerulus", "odor"]).to_dict()})
r4d = pd.DataFrame(rows)
print(r4d.to_string(index=False))
mt = {co: np.mean([traces[k][8][0] for k in traces if k[1] == "PGb" and k[2] == "3-hexanone" and meta.cohort[k[0]] == co], 0) for co in ("young", "old")}
plt.figure(figsize=(6, 3)); [plt.plot(np.arange(1, 41), v, label=k) for k, v in mt.items()]
plt.axvspan(9, 14, color="0.9"); plt.axvspan(35, 40, color="0.9"); plt.xlabel("frame"); plt.ylabel("PGb fold change (a)"); plt.legend(); plt.show()
print("Plotted range:", round(min(v.min() for v in mt.values()), 3), "to", round(max(v.max() for v in mt.values()), 3))

**Result: partially reproduced.** At the held-out placement (reading a, radius 8) the PGb response to
3-hexanone over frames 9 to 14 is larger in young ants (9.5 against 3.5, p = 0.0026; every variant
p <= 0.0061), but over frames 35 to 40 old ants are not above young ones (0.68 against 0.98,
p = 0.67; variants p 0.40 to 0.96). Data S2's own values give both (p = 0.0020 and 0.0098), so the
slower decline in old ants is not found in the dandiset's AUCs at the located ROIs.

### pg-opposite-modulation

> In PGb and PGa, responses to alarm pheromones rise with age while responses to general odorants
> fall or stay absent; no general odorant response rises with age in any focal glomerulus.

In [ ]:
res_all30 = run_all([(g, o) for g in GLOMS for o in antal.GENERAL], antal.GENERAL)
for k in res_all30:
    v = res_all30[k]
    up = v[(v.p < 0.05) & (v.mean_old > v.mean_young)]
    pg_down = v[v.glomerulus.isin(["PGb", "PGa"]) & (v.p < 0.05) & (v.mean_young > v.mean_old)]
    print(f"{k:>24}: general odorant tests with old significantly higher: {len(up)} of {len(v)}"
          + (f" ({', '.join(up.glomerulus + ' ' + up.odor)})" if len(up) else "")
          + f"; PGb/PGa general odorants declining: {len(pg_down)}")

**Result: not reproduced.** The criterion needs fig3d-pg-stronger-old, which is not reproduced, so
the claim is not. The general odorant parts hold: no general odorant response rises with age in any
of the six glomeruli (0 of 30 tests under every variant), and PGb and PGa responses to 3-hexanone
and isopropanol fall with age (4 of 4 significant at the held-out placement). On Data S2's own
values 2 of the 30 tests have old significantly "higher" (G2 and G3 with propionic acid, the
negative deflection nearer zero that fig4-propionic-acid-g2-g3 expects), so this criterion and
that claim's cannot both hold; the question is recorded for a reader.

In [ ]:
print(f"Notebook ran in {(time.time() - started) / 60:.1f} min")

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Data | Outcome |
|---|---|---|
| fig1c-old-faster-alarm | Data S1 | reproduced |
| fig1c-vehicle-single-curve | Data S1 | reproduced |
| fig1b-alarm-stimuli-similar | Data S1 | reproduced |
| figs1-heptanone-separate-curves | Data S1 | reproduced |
| figs1-heptanol-blend-single-curve | Data S1 | reproduced |
| fig2b-gcamp-baseline-unchanged | Brain Image Library | blocked: the download server was not reached |
| fig2d-whole-al-alarm-no-age-effect | dandiset | reproduced |
| figs3b-whole-al-general-decline | dandiset | partially reproduced (ratios) |
| figs3b-whole-al-other-no-age-effect | dandiset | reproduced |
| fig2e-heptanone-breadth-falls | dandiset | blocked: methods underspecified (ROI drawing) |
| fig2-alarm-sparse | dandiset | blocked: methods underspecified (depends on fig2e) |
| figs3c-general-breadth-no-age-effect | dandiset | blocked: methods underspecified (the same automated ROIs) |
| data-s2-glomerulus-link | dandiset and Data S2 | partially reproduced (66% agree; 41% on held-out trials) |
| figs2a-mean-six-glomeruli-no-age-effect | located ROIs | reproduced, resting on assumptions |
| fig3b-g2-heptanone-weaker-old | located ROIs | not reproduced |
| fig3d-pg-stronger-old | located ROIs | not reproduced |
| fig3d-pg-60-vs-90-days-same | located ROIs, DANDI 000467, Data S2 | blocked: methods underspecified (the readings disagree; 90-day PGa placements at chance; reading b mixes two pipelines) |
| figs4a-general-odorants-in-focal-glomeruli | located ROIs | not reproduced |
| fig4-general-odorants-decline | located ROIs | partially reproduced, resting on assumptions (not under the other variants) |
| fig4-propionic-acid-g2-g3 | located ROIs | not reproduced |
| figs4d-pgb-hexanone-dynamics | located ROIs | partially reproduced |
| pg-opposite-modulation | located ROIs | not reproduced |

How the analysis differs from the paper's. Frames are aligned by whole-pixel phase correlation,
not the ImageJ Image Stabilizer; the glomerulus traces use one set of shifts per trial, from the
middle plane. The AL outline is found by a threshold, not drawn. Responding ROIs are found by the
claims file's automated rule, at three glomerulus sizes, not drawn. The focal glomeruli are placed
where the dandiset's per-trial responses best follow Data S2's, with a disk ROI of 8 pixels radius;
the paper drew them by hand. The main measure reads each odor's AUCs where the glomerulus was placed
from the other odors' trials, so that the tested values did not choose the place; the placement
from all trials is shown as a variant. Fold change follows reading (a) (ROI mean, then fold change),
with reading (b) shown, leaving out pixels whose baseline is at background level. Each ant's value
is the mean over the trials Data S2 holds for that glomerulus (all trials are shown as a variant).

Pattern of the results. Everything the paper's own Data S1 and Data S2 values support was
recomputed above as a reference: the behaviour claims match Table S1 exactly, and Data S2's traces
give every focal glomerulus claim as the paper states it, except pg-opposite-modulation, whose
criterion fails on Data S2 (2 of 30 tests, G2 and G3 with propionic acid; asked of a reader). The
whole-AL measures, which here use a threshold outline in place of the paper's hand-drawn one, agree with the paper in direction and significance; the ratios of Figure S3B are
not reached, and the measure has a negative offset of about -0.2. Every young-ant glomerulus
placement is above chance, but 15 of 39 old-ant placements are not, and on held-out trials only 9
of 52 old-ant triples agree with Data S2. The focal glomerulus claims that rest on old ants' alarm
responses (Figure 3B and 3D) fail, and the AUCs measured here for old ants' PGb are well below
Data S2's in both this dandiset and DANDI 000467; so these outcomes say more about the located
ROIs than about the paper. Of the ten declines of general odorant responses the paper reports,
5 to 9 are significant here, depending on the placement and ROI. Where old-ant ROIs miss the
glomerulus, old values are read too low, which favours the claims that old responses are lower
(fig4) and works against those that they are higher (fig3d, figs4d).

What was found about the dataset. It holds the calcium imaging of all 21 ants the paper
describes, 22 trials each (with the paraffin oil control), and its ants and ages match Data S2.
It holds no ROI positions, glomerulus identities, stimulus times (only a description string) or
responding-ROI counts, so the focal glomerulus and ROI count analyses cannot be repeated from it
alone. Its timing metadata disagree (1.21 Hz series rate and a stored imaging rate of 0.83 against
1.2 s per volume in the description). One trial misspells ethylpyrazine in its file names. Raw
fluorescence is stored as float64, eight bytes per value, in 1.28 TB. The behaviour data are only
in the paper's supplement, the confocal stacks in the Brain Image Library, and the 90-day ants in
DANDI 000467.